# 04 · Validation against the analyst catalogue

An ML catalogue is a hypothesis until it is checked against independent observations. Here
we compare the day's automatic results with the **manually reviewed** ARGOS catalogue and its
analyst picks: 2,393 located events from 2012–2014 (Yemane et al.; files described in
`data/reference/README.md`).

Two levels of comparison:

1. **Picks.** For every analyst pick, did each model find it, and how far off in time was it?
   This isolates picker performance from association and location.
2. **Events.** Which reference events did we recover, which did we miss, and what did we add?
   How far apart are the locations and magnitudes?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tesfahiwet-yemane/SeismoML/blob/main/notebooks/04_catalogue_comparison.ipynb)

In [ ]:
# Running on Google Colab? This cell installs everything and gets the data (2-4 minutes).
# On your own computer it does nothing.
import os, subprocess, sys
if "google.colab" in sys.modules:
    repo = "/content/SeismoML"
    if not os.path.isdir(repo):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/tesfahiwet-yemane/SeismoML.git", repo], check=True)
    subprocess.run([sys.executable, f"{repo}/scripts/colab_setup.py"] + "--data --outputs".split(), check=True)
    sys.path.insert(0, f"{repo}/src")
    os.chdir(f"{repo}/notebooks")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from obspy import UTCDateTime

import seismoml as sml
from seismoml import catalogue, data, magnitude, picking, plotting

plotting.use_style()
cfg = sml.load_config()

ref_cat_all = catalogue.read_catalogue(cfg.path("reference_catalogue"))
ref_picks_all = catalogue.read_reference_picks(cfg.path("reference_picks"))
ref_cat = catalogue.select_window(ref_cat_all, cfg.t0, cfg.t1)
ref_picks = catalogue.select_window(ref_picks_all, cfg.t0, cfg.t1)
ref_picks = ref_picks[ref_picks["station"].isin(cfg.stations)].reset_index(drop=True)
print(f"Reference catalogue: {len(ref_cat_all)} events in total, {len(ref_cat)} on {cfg.day.date}")
print(f"Reference picks on {cfg.day.date}: {len(ref_picks)} "
      f"({(ref_picks.phase == 'P').sum()} P, {(ref_picks.phase == 'S').sum()} S)")
ref_cat

The reference catalogue has no magnitudes, so we compute them with **exactly** the procedure
used for the ML catalogues in notebook 03 (Keir et al., 2006, from the analyst picks). This
way magnitude differences reflect the picks and locations, not two different scales.

In [ ]:
stations = data.active_stations(data.station_table(cfg.path("stations")), cfg.day)
if ref_cat["magnitude"].isna().all():
    inventory = data.load_inventory(cfg.inventory_file)
    ref_cat, ref_amps = magnitude.catalogue_magnitudes(
        ref_cat, ref_picks, stations, inventory,
        lambda sta: data.load_waveforms(cfg.sds, cfg.network, sta, cfg.t0, cfg.t1))
ref_cat[["event_id", "time", "latitude", "longitude", "depth_km", "magnitude"]]

## 1 · Pick-level comparison

Analysts pick only phases that belong to catalogued events, so a model pick with no analyst
counterpart is *not necessarily* a false alarm. We therefore report

* **recall**: the fraction of analyst picks recovered within ±0.5 s, and
* **timing residuals** (model − analyst) of the recovered picks.

A *precision proxy* is computed only where the analyst actually looked: for each event and
station with analyst picks, from 2 s before the first to 2 s after the last pick. It is a
lower bound, because an analyst may skip a phase the model picks correctly.

In [ ]:
labels = ["PN-original", "PN-instance", "PN-stead", "PN-volpick", "EQT-original", "EQT-volpick"]
ml_picks = {k: picking.read_picks(cfg.outputs / "picks" / f"{k}.csv") for k in labels}

windows = ref_picks.groupby(["event_id", "station"])["time"].agg(["min", "max"]).reset_index()


def in_windows(df: pd.DataFrame) -> pd.DataFrame:
    """Model picks inside an analyst event-station window (±2 s)."""
    keep = np.zeros(len(df), dtype=bool)
    pad = pd.Timedelta(seconds=2)
    for _, w in windows.iterrows():
        keep |= ((df["station"] == w["station"]) &
                 (df["time"] >= w["min"] - pad) & (df["time"] <= w["max"] + pad)).to_numpy()
    return df[keep]

scores = []
matches = {}
for label, df in ml_picks.items():
    s = picking.detection_scores(ref_picks, in_windows(df), tolerance=0.5)
    s["model"] = label
    scores.append(s.reset_index())
    matches[label] = picking.match_picks(ref_picks, df, tolerance=0.5)
scores = pd.concat(scores).set_index(["model", "phase"])
scores.style.format(precision=3, na_rep="–")

In [ ]:
fig = plotting.residual_histograms(matches, tolerance=0.5)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
x = np.arange(len(labels))
for ax, ph in zip(axes, "PS"):
    sub = scores.xs(ph, level="phase").loc[labels]
    ax.bar(x - 0.2, sub["recall"], 0.4, label="recall", color="#4c72b0")
    ax.bar(x + 0.2, sub["precision"], 0.4, label="precision proxy", color="#dd8452")
    ax.set_xticks(x, labels, rotation=35, ha="right")
    ax.set_title(f"{ph} phases vs. analyst picks")
    ax.set_ylim(0, 1)
axes[0].legend(loc="lower left");

### Recall by station

Station-dependent recall exposes site effects. Noisy sites near the geothermal plant or
villages, and stations on altered ground, behave differently.

In [ ]:
rec_sta = {}
for label in ("PN-volpick", "PN-instance", "EQT-volpick"):
    m = matches[label]
    rec_sta[label] = (m.groupby("station").size() /
                      ref_picks.groupby("station").size()).reindex(cfg.stations)
rec_sta = pd.DataFrame(rec_sta)
fig = plotting.matrix_heatmap(rec_sta.T, title="Recall of analyst picks (P+S) by station")

## 2 · Event-level comparison

Events are matched one-to-one when origin times differ by at most 3 s and hypocentres by at
most 10 km.

In [ ]:
cats = {}
for label in ("PN-volpick", "PN-instance", "EQT-volpick"):
    cats[label] = catalogue.read_catalogue(cfg.outputs / "catalogues" / f"{label}_events.csv")

event_rows, ev_matches = [], {}
for label, cat in cats.items():
    m = catalogue.match_events(ref_cat, cat, max_dt=3.0, max_dist_km=10.0, epsg=cfg.utm_epsg)
    ev_matches[label] = m
    event_rows.append({
        "catalogue": label,
        "events": len(cat),
        "matched": len(m),
        "recall": len(m) / max(len(ref_cat), 1),
        "new (unmatched)": len(cat) - len(m),
        "median |Δt| (s)": m["dt_s"].abs().median(),
        "median Δh (km)": m["dh_km"].median(),
        "median Δz (km)": m["dz_km"].median(),
    })
pd.DataFrame(event_rows).set_index("catalogue").style.format(precision=2)

In [ ]:
fig = plotting.map_and_sections(
    stations,
    {"reference": ref_cat, **cats}, cfg.lat_range, cfg.lon_range, cfg.depth_range,
    title=f"Reference vs. ML catalogues, {cfg.day.date}")

### Location and magnitude differences for matched events

In [ ]:
label = "PN-volpick"
m = ev_matches[label]
if len(m):
    mm = m.merge(ref_cat[["event_id", "magnitude", "depth_km"]], left_on="ref_id", right_on="event_id") \
          .merge(cats[label][["event_id", "magnitude"]], left_on="cand_id", right_on="event_id",
                 suffixes=("_ref", "_ml"))
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
    axes[0].hist(m["dh_km"], bins=20, color="#4c72b0")
    axes[0].set(xlabel="Epicentral difference (km)", ylabel="Events")
    axes[1].scatter(mm["depth_km"], mm["depth_km"] + mm["dz_km"], s=18, edgecolor="k", lw=0.3)
    lim = [cfg.depth_range[0], cfg.depth_range[1]]
    axes[1].plot(lim, lim, "k--", lw=0.8)
    axes[1].set(xlabel="Reference depth (km)", ylabel=f"{label} depth (km)", xlim=lim, ylim=lim)
    ok = np.isfinite(mm["magnitude_ref"]) & np.isfinite(mm["magnitude_ml"])
    axes[2].scatter(mm.loc[ok, "magnitude_ref"], mm.loc[ok, "magnitude_ml"], s=18, edgecolor="k", lw=0.3)
    if ok.sum() > 1:
        d = (mm.loc[ok, "magnitude_ml"] - mm.loc[ok, "magnitude_ref"])
        axes[2].set_title(f"ΔML = {d.mean():+.2f} ± {d.std():.2f}")
    mlim = [np.nanmin(mm[["magnitude_ref", "magnitude_ml"]].to_numpy()) - 0.2,
            np.nanmax(mm[["magnitude_ref", "magnitude_ml"]].to_numpy()) + 0.2] if ok.any() else [-1, 3]
    axes[2].plot(mlim, mlim, "k--", lw=0.8)
    axes[2].set(xlabel="Reference ML", ylabel=f"{label} ML", xlim=mlim, ylim=mlim)

### Are the "new" events real?

ML catalogues typically contain many more events than manual ones. Before celebrating,
inspect them. Below is the best-recorded PN-volpick event with no reference counterpart.

In [ ]:
new = cats[label][~cats[label]["event_id"].isin(ev_matches[label]["cand_id"])]
asg = picking.read_picks(cfg.outputs / "catalogues" / f"{label}_assigned_picks.csv")
if len(new):
    e = new.assign(n=new["n_P"] + new["n_S"]).sort_values("n", ascending=False).iloc[0]
    t_e = UTCDateTime(e["time"].timestamp())
    st = data.load_waveforms(cfg.sds, cfg.network, cfg.stations, t_e - 5, t_e + 20)
    st.detrend("demean").filter("bandpass", freqmin=2, freqmax=25, zerophase=True)
    st_xy = catalogue.to_utm(stations, cfg.utm_epsg)
    e_xy = catalogue.to_utm(pd.DataFrame([e]), cfg.utm_epsg).iloc[0]
    dist = {r.station: float(np.hypot(r.easting_km - e_xy.easting_km, r.northing_km - e_xy.northing_km))
            for r in st_xy.itertuples()}
    fig = plotting.record_section(st, asg[asg["event_id"] == e["event_id"]], dist, t_e - 2, t_e + 12,
                                  title=f"New event {e['time']:%H:%M:%S}, ML {e['magnitude']:.1f}, "
                                        f"{int(e['n_P'])} P + {int(e['n_S'])} S picks")

## Take-aways

* Recall of analyst picks and timing bias are the most direct measures of picker quality on a
  new volcano.
* Unmatched ML events are a mix of real small events (below the analyst's detection threshold)
  and false associations. Visual inspection, or a stricter `n_picks`, separates them.

## Exercises

1. Compare recall for the reference events as a function of their magnitude. At what ML do
   the models start to miss events?
2. Repeat the event matching with `max_dt=1.5`. How sensitive are the numbers?
3. Use the analyst picks of matched events to estimate a Vp/Vs ratio with a Wadati diagram,
   and do the same with the PN-volpick picks. Do they agree?

**Next:** 05 · Seismicity analysis (`05_seismicity_analysis.ipynb`)